# LSTM Training with 5-Fold CV and Extreme Anti-Overfitting

In [ ]:
import os
import glob
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Matplotlib Publication Standards
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'lines.linewidth': 2.0,
    'grid.alpha': 0.3
})
sns.set_style("whitegrid")


In [ ]:
DATASET_DIR = r"D:\Proyek Dosen\Riset Bearing\XJTU-SY_Bearing_Datasets\Processed_Data\LSTM_Inputs"
RESULTS_DIR = r"D:\Proyek Dosen\Riset Bearing\Training_Results"
os.makedirs(RESULTS_DIR, exist_ok=True)

WINDOW_SIZES = [10, 20, 30]
FOLDS = 5
NUM_FEATURES = 14


'''
---------------------------------------------------------------------------
Training Phase Reference (apply in 3_TRAINING_LSTM.ipynb):
  NOISE_STD    = 0.0    (disable Gaussian noise entirely)
  DROPOUT      = 0.1    (halved from 0.2)
  WEIGHT_DECAY = 1e-5   (reduced 10x from 1e-4)
  HIDDEN_DIM   = 128    (doubled from 64)
  LEARNING_RATE = 0.0005
---------------------------------------------------------------------------
'''

# Scaled Capacity Hyperparameters for Segmented Data
EPOCHS = 60
BATCH_SIZE = 128
LEARNING_RATE = 0.0005
WEIGHT_DECAY = 1e-5
HIDDEN_DIM = 128
NUM_LAYERS = 2
NOISE_STD = 0.0
DROPOUT = 0.1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Computation Device: {DEVICE}")

In [ ]:
def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def scale_health_index_to_rul(hi, max_rul):
    return np.clip(hi, 0, 1) * max_rul


In [ ]:
class RobustModel(nn.Module):
    """
    High-capacity Vanilla LSTM Network for RUL Prediction.
    Adapted for high-volume segmented sequence data natively avoiding data starvation.
    """
    def __init__(self, input_size: int = 14, hidden_dim: int = 64, num_layers: int = 2, output_size: int = 1, dropout: float = 0.2, noise_std: float = 0.01):
        super(RobustModel, self).__init__()
        self.noise_std = noise_std
        
        # Scaled-up Recurrent Layers natively implementing dropout across stacks
        self.rnn = nn.LSTM(input_size, hidden_dim, num_layers, batch_first=True, dropout=(dropout if num_layers > 1 else 0))
        
        # Regularization
        self.dropout = nn.Dropout(dropout)
        
        # Regression Head (STRICTLY NO RELU)
        self.linear = nn.Linear(hidden_dim, output_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Subtle Gaussian Noise Injection for robust feature learning (Train only)
        if self.training and self.noise_std > 0.0:
            x = x + torch.randn_like(x) * self.noise_std
            
        out, _ = self.rnn(x)
        
        # Extract the last time step representing Target Y chronologically
        out = self.dropout(out[:, -1, :])
        
        # Raw linear output to prevent dead gradient bounds (Dying ReLU avoided)
        return self.linear(out)

## 5. Model Evaluation and Visualization Setup
Initialize publication-standard visualization settings and helper functions for result analysis.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Publication-standard plotting style configuration
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif', 'serif'],
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'axes.grid.which': 'both',
    'grid.color': '#bfbfbf',
    'grid.alpha': 0.35,
    'grid.linestyle': '--',
})

def plot_health_index(val_df, ws, fold):
    """
    Plots the Ground Truth vs Smoothed Predicted Health Index.
    Handles multiple bearings gracefully by plotting them sequentially.
    """
    if 'Bearing_ID' in val_df.columns:
        bearings = val_df['Bearing_ID'].unique()
    else:
        bearings = ['Unknown_Bearing']
    
    fig, axes = plt.subplots(len(bearings), 1, figsize=(10, 4 * len(bearings)), squeeze=False)
    
    for idx, bearing in enumerate(bearings):
        ax = axes[idx, 0]
        
        b_df = val_df[val_df['Bearing_ID'] == bearing].copy()
            
        if 'Original_Minute' in b_df.columns:
            b_df = b_df.sort_values(by='Original_Minute')
            x_axis = b_df['Original_Minute']
        else:
            b_df = b_df.reset_index()
            x_axis = b_df.index
            
        ax.plot(x_axis, b_df['True_HI'], label='Ground Truth (HI)', color='#1f77b4', linewidth=2)
        
        pred_col = 'Smoothed_Predicted_HI' if 'Smoothed_Predicted_HI' in b_df.columns else 'Predicted_HI'
        ax.plot(x_axis, b_df[pred_col], label='LSTM Prediction (Smoothed)', color='#d62728', linestyle='--', linewidth=2.0)
        
        ax.set_title(f'LSTM RUL Prediction - Validation (Window Size: {ws}, Fold: {fold} | Bearing: {bearing})')
        ax.set_xlabel('Operating Time (Minutes)')
        ax.set_ylabel('Health Index (HI)')
        ax.set_ylim([-0.05, 1.05])
        ax.legend(loc='upper right', frameon=True, framealpha=0.9, edgecolor='black')
        
    plt.tight_layout()
    plt.show()


## 6. Model Training & Cross-Fold Evaluation
Train the LSTM model across various window sizes using Stratified 5-Fold Cross Validation.
Predictions and performance markers are cached for downstream reporting.

In [ ]:
summary_metrics = []
global_results_dir = "LSTM_Results"
os.makedirs(global_results_dir, exist_ok=True)

print("Starting LSTM Training & Evaluation Pipeline...")

for ws in WINDOW_SIZES:
    print(f"\n{'='*60}\nEvaluating Window Size: {ws}\n{'='*60}")
    ws_dir = os.path.join(DATASET_DIR, f"window_size_{ws}")
    
    # Target directory for saving results of this window size
    output_ws_dir = os.path.join(global_results_dir, f"window_size_{ws}")
    os.makedirs(output_ws_dir, exist_ok=True)
    
    fold_metrics = []
    ws_all_predictions = pd.DataFrame()
    
    for fold in range(1, FOLDS + 1):
        print(f"\n--- Fold {fold} Processing ---")
        train_file = os.path.join(ws_dir, f"processed_train_fold{fold}.csv")
        val_file = os.path.join(ws_dir, f"processed_val_fold{fold}.csv")
        
        if not os.path.exists(train_file) or not os.path.exists(val_file):
            print(f"[Warning] Missing data for Fold {fold}. Skipping this fold.")
            continue
            
        df_train = pd.read_csv(train_file)
        df_val = pd.read_csv(val_file)
        
        # Extract true targets
        y_train = df_train['Health_Index'].values
        y_val = df_val['Health_Index'].values
        
        # Scrambling Prevention Pipeline
        drop_cols = [c for c in ['Health_Index', 'Target_RUL', 'Bearing_ID', 'Change_Point', 'Original_Minute'] if c in df_train.columns]
        
        # Filter strictly numeric to avoid accidental object array type casting
        df_train_num = df_train.select_dtypes(include=[np.number])
        df_val_num = df_val.select_dtypes(include=[np.number])
        
        x_train_flat = df_train_num.drop(columns=[c for c in drop_cols if c in df_train_num.columns]).values.astype(np.float32)
        x_val_flat = df_val_num.drop(columns=[c for c in drop_cols if c in df_val_num.columns]).values.astype(np.float32)
        
        actual_num_features = x_train_flat.shape[1] // ws
        
        # Matrix reconstruction back to Sequence Topology
        x_train_3d = x_train_flat.reshape(-1, actual_num_features, ws).transpose(0, 2, 1)
        x_val_3d = x_val_flat.reshape(-1, actual_num_features, ws).transpose(0, 2, 1)
        
        # Convert to PyTorch Tensors
        tensor_x_train = torch.tensor(x_train_3d, dtype=torch.float32).to(DEVICE)
        tensor_y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1).to(DEVICE)
        tensor_x_val = torch.tensor(x_val_3d, dtype=torch.float32).to(DEVICE)
        
        train_loader = DataLoader(TensorDataset(tensor_x_train, tensor_y_train), batch_size=BATCH_SIZE, shuffle=True)
        
        model = RobustModel(input_size=actual_num_features, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, noise_std=NOISE_STD, dropout=DROPOUT).to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        criterion = nn.MSELoss()
        
        # Start Routine Training
        model.train()
        for epoch in range(EPOCHS):
            for batch_x, batch_y in train_loader:
                optimizer.zero_grad()
                pred = model(batch_x)
                loss = criterion(pred, batch_y)
                loss.backward()
                optimizer.step()
                
        # Routine Evaluation (Batched Inference to Prevent OOM)
        model.eval()
        pred_hi_list = []
        
        # Instantiate validation DataLoader without shuffling
        val_loader = DataLoader(tensor_x_val, batch_size=BATCH_SIZE, shuffle=False)
        
        with torch.no_grad():
            for batch_x in val_loader:
                batch_pred = model(batch_x).cpu().numpy().flatten()
                pred_hi_list.append(batch_pred)
                
            pred_hi_concat = np.concatenate(pred_hi_list)
            pred_hi = np.clip(pred_hi_concat, 0, 1) # Guardrail clipping

            # Formulate Validation DataFrame for Grouping
            val_results = pd.DataFrame()
            if 'Bearing_ID' in df_val.columns:
                val_results['Bearing_ID'] = df_val['Bearing_ID'].values
            else:
                val_results['Bearing_ID'] = 'Unknown'
                
            if 'Original_Minute' in df_val.columns:
                val_results['Original_Minute'] = df_val['Original_Minute'].values
            elif 'Minute' in df_val.columns:
                val_results['Original_Minute'] = df_val['Minute'].values
            else:
                val_results['Original_Minute'] = np.arange(len(df_val)) // 63 + 1
                
            if 'Change_Point' in df_val.columns:
                val_results['Change_Point'] = df_val['Change_Point'].values
                
            val_results['True_HI'] = y_val
            val_results['Predicted_HI'] = pred_hi
            val_results['Fold'] = fold
            val_results['Window_Size'] = ws
            
            # 1. Group By Minute (Ensemble Averaging)
            smoothed_results = val_results.groupby(['Bearing_ID', 'Original_Minute'], as_index=False).agg({
                'True_HI': 'mean',
                'Predicted_HI': 'mean',
                'Fold': 'first',
                'Window_Size': 'first'
            })
            
            # 2. Exponential Smoothing (EMA)
            smoothed_dfs = []
            for b_id, b_df in smoothed_results.groupby('Bearing_ID'):
                b_df = b_df.sort_values('Original_Minute').copy()
                b_df['Smoothed_Predicted_HI'] = b_df['Predicted_HI'].ewm(span=5, adjust=False).mean()
                smoothed_dfs.append(b_df)
            smoothed_results = pd.concat(smoothed_dfs, ignore_index=True)
            
            # 3. Recalculate Metrics (Macroscopic)
            rmse_val = calculate_rmse(smoothed_results['True_HI'], smoothed_results['Smoothed_Predicted_HI'])
            mae_val = mean_absolute_error(smoothed_results['True_HI'], smoothed_results['Smoothed_Predicted_HI'])
            r2_val = r2_score(smoothed_results['True_HI'], smoothed_results['Smoothed_Predicted_HI'])
            
            fold_metrics.append({
                'Fold': fold,
                'RMSE': rmse_val,
                'MAE': mae_val,
                'R2': r2_val
            })
            print(f"[Fold {fold} Smoothed Results] -> RMSE: {rmse_val:.4f} | MAE: {mae_val:.4f} | R2: {r2_val:.4f}")
            
            ws_all_predictions = pd.concat([ws_all_predictions, smoothed_results], ignore_index=True)
            
            # 4. Refactor Plotting Logic (Plotted automatically using Operating Time)
            if fold == 1: 
                plot_health_index(smoothed_results, ws, fold)
                
        # Save model from the final fold natively as proxy artifact
        if fold == FOLDS:
            torch.save(model.state_dict(), os.path.join(output_ws_dir, 'optimized_lstm_model.pth'))
                
    if fold_metrics:
        # Export fold markers within the respective window size directory
        df_eval = pd.DataFrame(fold_metrics)
        df_eval.to_csv(os.path.join(output_ws_dir, f"evaluation_markers_w{ws}.csv"), index=False)
        
        # Export holistic prediction mappings
        if not ws_all_predictions.empty:
            ws_all_predictions.to_csv(os.path.join(output_ws_dir, f"prediction_states_w{ws}.csv"), index=False)
        
        avg_rmse = df_eval['RMSE'].mean()
        avg_mae = df_eval['MAE'].mean()
        avg_r2 = df_eval['R2'].mean()
        print(f"\n==> Global Performance for Window Size {ws}:")
        print(f"    Average RMSE: {avg_rmse:.4f}")
        print(f"    Average MAE: {avg_mae:.4f}")
        print(f"    Average R2: {avg_r2:.4f}")
        
        summary_metrics.append({
            'Window_Size': str(ws), 
            'Avg_RMSE': avg_rmse,
            'Avg_MAE': avg_mae,
            'Avg_R2': avg_r2
        })

# Store high-level comparison framework
if summary_metrics:
    summary_df = pd.DataFrame(summary_metrics)
    summary_csv_path = os.path.join(global_results_dir, "LSTM_Global_Window_Comparison.csv")
    summary_df.to_csv(summary_csv_path, index=False)
    
    print("\n" + "="*60)
    print("FINAL EVALUATION ACROSS ALL WINDOW SIZES")
    print("="*60)
    print(summary_df.to_string(index=False))
else:
    print("\n[Warning] No metrics collected during iteration. Review dataset pipelines.")

## 7. Hyperparameter Analysis: Average RMSE Comparison
Bar chart representation to natively select the lowest-RMSE configuration across variable temporal steps, adhering to scientific publication benchmarks.

In [ ]:
if 'summary_df' in locals() and not summary_df.empty:
    plt.figure(figsize=(9, 5))
    ax = sns.barplot(
        x='Window_Size', 
        y='Avg_RMSE', 
        data=summary_df, 
        palette='viridis', 
        edgecolor='black',
        linewidth=1.2
    )

    # Insert scalar annotations directly onto diagram
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.4f}", 
                    (p.get_x() + p.get_width() / 2., p.get_height()), 
                    ha='center', va='bottom', 
                    xytext=(0, 6), 
                    textcoords='offset points',
                    fontsize=11,
                    fontweight='bold')

    plt.title('Average Validation RMSE Comparison Across Window Sizes (LSTM Native)', pad=15)
    plt.xlabel('Window Size (Timesteps)')
    plt.ylabel('Average Validation RMSE')
    
    # Scale max Y headroom strictly
    max_y = summary_df['Avg_RMSE'].max()
    plt.ylim([0, max_y * 1.25]) 
    
    plt.tight_layout()
    plt.savefig(os.path.join(global_results_dir, 'LSTM_RMSE_Comparison_BarChart.png'), dpi=300)
    plt.show()
else:
    print("No summary data available for plotting. Execute the training loop first.")